# FlowGuard - EDA & Preprocessing
Explore data, run preprocessing pipeline.

In [ ]:
import os, sys

# ── Colab setup: clone repo and set working directory ─────────────────────────
REPO_DIR = "/content/flowguard"
if not os.path.exists(REPO_DIR):
    raise RuntimeError("Run notebook 00_setup_and_validate.ipynb first to clone the repo and copy datasets.")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.config import load_config
from src.utils.reproducibility import setup_reproducibility

config = load_config("configs/base.yaml")
setup_reproducibility(config)
print("Config loaded. Datasets:", [d['name'] for d in config['data']['datasets']])

## Quick EDA on Raw Data

In [ ]:
# Sample from UNSW for EDA
sample = pd.read_csv("data/raw/NF-UNSW-NB15-v3.csv", nrows=50000)
print(f"Shape: {sample.shape}")
print(f"\nColumns: {list(sample.columns)}")
print(f"\nLabel distribution:\n{sample['Label'].value_counts()}")
print(f"\nAttack types:\n{sample['Attack'].value_counts()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, ['IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'PROTOCOL']):
    if col in sample.columns:
        sample[col].hist(ax=ax, bins=50)
        ax.set_title(col)
plt.tight_layout()
plt.savefig("results/plots/raw_distributions.png", dpi=100)
plt.show()

## Run Preprocessing Pipeline

In [ ]:
from src.data.preprocess import run_full_preprocessing

# Runs the full pipeline: CSV → Parquet + combined_unlabeled.parquet
# Reads dataset paths from DATASET_CATALOG in preprocess.py (matches data/raw/ filenames)
all_stats = run_full_preprocessing("configs/base.yaml")

print(f"\nProcessed {len(all_stats)} datasets.")
for name, stats in all_stats.items():
    print(f"  {name}: {len(stats.feature_names)} features")

In [ ]:
# combined_unlabeled.parquet is already created by run_full_preprocessing above
combined_path = "data/processed/combined_unlabeled.parquet"
if os.path.exists(combined_path):
    import pandas as pd
    df = pd.read_parquet(combined_path)
    print(f"Combined unlabeled: {len(df):,} rows, {df.shape[1]} features")
else:
    print("combined_unlabeled.parquet not found — check that at least one dataset processed successfully.")

In [ ]:
from src.data.splits import generate_all_splits
generate_all_splits("configs/base.yaml")

In [ ]:
for name in ['unsw', 'bot', 'ton', 'cic']:
    path = f"data/processed/{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"{name}: {len(df):,} rows, {df.shape[1]} cols")
        print(f"  Label dist: {df['Label'].value_counts().to_dict()}")